# 獲得関数の最適化：候補提案と制約付き探索

学習済みGPと獲得関数から**次の実験条件を選ぶ方法**を学びます。`robotorchan.optim.optimize_acqf` を使って、連続最適化、joint `q=2`、固定変数、線形制約、候補間制約、Sobol探索を比較します。

さらに、候補の観測 → `fit()` によるGP再学習 → 次の候補提案まで実行します。既存のBoTorch・robotorchan公開APIを優先し、独自の獲得関数や最適化器は追加しません。


## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement

from robotorchan.models import SingleTaskGP
from robotorchan.optim import (
    CandidateConstraints,
    get_optimizer_capabilities,
    optimize_acqf,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### モデル・獲得関数・最適化器の責務

- **GPモデル**は入力に対する応答と不確実性を予測します。
- **獲得関数**は予測を使い「どの入力を測る価値が高いか」を数値化します。
- **最適化器**は指定した探索空間で獲得関数を最大化し、次の候補入力を返します。

この3つは独立した役割です。最適化器をSobolに変更しても、学習済みGPや獲得関数の定義は変わりません。

## 2. Fit a surrogate and construct the acquisition

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    x0 = X[..., :1]
    x1 = X[..., 1:2]
    return torch.sin(2 * torch.pi * x0) + 0.35 * torch.cos(2 * torch.pi * x1)


train_X = torch.rand(14, 2)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

## 3. Canonical continuous optimization

default `optimizer="botorch"`はBoTorch/SciPy multi-start optimizationへ薄く委譲します。BoTorchのacquisition vocabularyを置き換えません。

### 連続最適化のパラメータ

`bounds` は各入力変数の下限・上限です。`q=1` は次に測る入力が1点であることを意味します。`num_restarts=4` は局所最適化の開始点の数、`raw_samples=64` は初期点選択に利用する候補数です。

`value` は獲得関数の値であり、実験で観測する応答 `Y` ではありません。

In [ ]:
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, 2])
print("candidate:", candidate)
print("acquisition value:", value)

## 4. Joint q-batch optimization

`q=2`では`[2, d]`全体が1つのdecision variableです。「q=1 optimizerを2回呼ぶ」こととは同じではありません。

### `q=2` は2点をまとめて評価する

`qLogExpectedImprovement` は候補2点の**組合せによる改善**を評価します。単独の候補を2回選ぶ場合とは異なり、2点の相関や重複した情報を考慮できます。

出力の `[2,2]` は **2候補 × 2入力変数** を意味します。2つの目的関数を同時に最適化する多目的BOとは別の概念です。

In [ ]:
batch_candidate, batch_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert batch_candidate.shape == torch.Size([2, 2])
print("joint q=2 candidate:\n", batch_candidate)
print("joint acquisition value:", batch_value)

## 5. Fixed features

既知coordinateを固定することはmodelやobjectiveの責務ではなくcandidate optimizationのcontractです。

### 固定変数は探索条件として指定する

`fixed_features={1:0.4}` は2番目の入力変数 `x1` を0.4に固定し、`x0` のみを探索する設定です。製造条件の一部を変更できない場合などに利用できます。

固定値はモデルの予測式を変更するものではなく、**候補探索の可行領域**を限定します。

In [ ]:
fixed_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    fixed_features={1: 0.4},
)

torch.testing.assert_close(fixed_candidate[:, 1], torch.tensor([0.4]))
print("fixed-feature candidate:", fixed_candidate)

## 6. Candidate-space linear constraints

`CandidateConstraints`は既知の入力制約です。BoTorch conventionに合わせ、不等式は`sum(coefficients * X[indices]) >= rhs`です。ここでは`x0 + x1 >= 1.1`を課します。

### 入力制約と出力制約を区別する

`CandidateConstraints` は入力 `X` に対して**既知の制約**を課します。例では `x0+x1 >= 1.1` の範囲だけを探索します。

一方、未知の品質制約をGPで予測する場合は**出力側の確率的制約**であり、この例とは異なる扱いが必要です。

In [ ]:
constraint = (
    torch.tensor([0, 1], dtype=torch.long),
    torch.tensor([1.0, 1.0]),
    1.1,
)
constraints = CandidateConstraints(
    inequality_constraints=(constraint,),
)

constrained_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    constraints=constraints,
)

assert constrained_candidate.sum() >= 1.1 - 1e-6
print("constrained candidate:", constrained_candidate)

## 7. Inter-point constraint requires joint q semantics

inter-point constraintはcandidate row単独では評価できません。ここではq=2の2候補について`x0(candidate 0) + x0(candidate 1) >= 1.2`を課します。

### 候補間制約はbatch全体で判定する

`joint_indices` の各行は `[候補番号, 入力変数番号]` です。今回の `[[0,0],[1,0]]` は2候補それぞれの `x0` を参照し、その和が1.2以上となるよう制約します。

これは各候補を独立に最適化するだけでは表現できない、**joint q-batch特有の制約**です。

In [ ]:
joint_indices = torch.tensor([[0, 0], [1, 0]], dtype=torch.long)
joint_coefficients = torch.tensor([1.0, 1.0])
joint_constraints = CandidateConstraints(
    inequality_constraints=((joint_indices, joint_coefficients, 1.2),),
)

joint_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    constraints=joint_constraints,
)

assert joint_candidate[:, 0].sum() >= 1.2 - 1e-6
print("joint constrained candidates:\n", joint_candidate)

## 8. Backend capability is part of the runtime contract

backend名だけで互換と判断せず、variable type、constraints、q、sequential、fixed featuresなどのcapabilityを確認します。

### Backend capabilityを確認する理由

`get_optimizer_capabilities()` は最適化器が連続変数、q-batch、固定変数、線形制約などを扱えるか確認するための公開APIです。

同じ獲得関数を渡せても、すべての最適化器が同じ制約を扱えるとは限りません。未対応の組合せを黙って近似せず、**対応表を確認して選択**します。

In [ ]:
for name in ["botorch", "sobol", "de", "ga", "pso"]:
    capabilities = get_optimizer_capabilities(name)
    print(
        name,
        {
            "continuous": capabilities.continuous,
            "q_batch": capabilities.q_batch,
            "fixed_features": capabilities.fixed_features,
            "linear_constraints": capabilities.linear_inequality_constraints,
        },
    )

## 9. A gradient-free backend keeps the acquisition contract

backendを変えても入力は同じBoTorch `AcquisitionFunction`です。ここでは軽量なSobol searchを比較します。

### Sobol探索と勾配ベース最適化の違い

BoTorch/SciPyのマルチスタートは局所的な連続最適化を利用します。Sobol探索は低食い違い系列で候補を評価し、勾配を利用しない探索の例です。

両者は**同じ獲得関数**を最大化しますが、探索アルゴリズムと計算コストが異なるため、提案点が一致するとは限りません。

In [ ]:
sobol_candidate, sobol_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="sobol",
    raw_samples=256,
    seed=123,
)

assert sobol_candidate.shape == torch.Size([1, 2])
print("Sobol-search candidate:", sobol_candidate)
print("Sobol-search acquisition value:", sobol_value)

## 10. Candidate evaluation closes the optimization layer

### 提案された候補を実際に観測する

`optimize_acqf` が返すのは次に測定する入力です。ここでは人工関数 `response()` を実験装置の代わりに使い、新しい応答値を取得します。

候補を選ぶことと観測データを増やすことは別です。**観測結果を追加して再学習したとき**、GPの予測が更新されます。

In [ ]:
candidate_Y = response(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate Y:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

### 観測後に再学習し、次の候補を提案する

新しいデータを追加したGPで `fit()` を実行し、更新した `best_f` を使って獲得関数を作り直します。これが通常のベイズ最適化の1反復です。

In [ ]:
updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
next_acquisition = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_Y.max(),
)
next_candidate, next_value = optimize_acqf(
    acq_function=next_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)
print("New observed response:", candidate_Y.item())
print("Updated best observed response:", updated_Y.max().item())
print("Next candidate:", next_candidate)
print("Next acquisition value:", next_value)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert next_candidate.shape == (1, 2)

### GPの予測更新を2次元で可視化する

2次元の入力領域を格子状に評価し、再学習前後のGP事後平均を並べて表示します。最初の観測点と追加した候補も重ね、どの入力を測ったか確認します。

In [ ]:
import matplotlib.pyplot as plt

axis = torch.linspace(0.0, 1.0, 41)
mesh_x0, mesh_x1 = torch.meshgrid(axis, axis, indexing="ij")
plot_X = torch.stack([mesh_x0.reshape(-1), mesh_x1.reshape(-1)], dim=-1)
with torch.no_grad():
    before_mean = model.posterior(plot_X).mean.reshape(41, 41)
    after_mean = updated_model.posterior(plot_X).mean.reshape(41, 41)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
levels = torch.linspace(
    torch.minimum(before_mean.min(), after_mean.min()).item(),
    torch.maximum(before_mean.max(), after_mean.max()).item(),
    18,
).numpy()
for ax, surface, title in (
    (axes[0], before_mean, "Before observation"),
    (axes[1], after_mean, "After observation and fit"),
):
    contour = ax.contourf(mesh_x0.numpy(), mesh_x1.numpy(), surface.numpy(), levels=levels)
    ax.scatter(train_X[:, 0], train_X[:, 1], s=16, label="Initial observations")
    ax.scatter(candidate[:, 0], candidate[:, 1], marker="x", s=80, label="New observation")
    ax.set(xlabel="x0", ylabel="x1", title=title)
    ax.legend(fontsize=7)
fig.colorbar(contour, ax=axes, label="Posterior mean")
plt.show()

## 11. Selection map

| Situation | Optimization path |
| --- | --- |
| smooth continuous acquisition | BoTorch multi-start |
| joint q-batch | q-aware backend; preserve `[q, d]` semantics |
| known candidate constraints | constraint-capable backend |
| integer/categorical/mixed | structured-space capable backend |
| unavailable candidate gradients | derivative-free / sampling backend |
| one-shot qKG/qMFKG | one-shot-aware optimization path |
| high-dimensional embedding | search strategy then backend |
| TuRBO | trust-region strategy then backend / Thompson path |
| vector numerical objective | specialist optimizer such as NSGA-II when appropriate |

## 12. Important boundaries

- acquisition optimizationはsurrogate modelを再定義しません。
- search strategyとnumerical optimizer backendは別レイヤーです。
- candidate/input constraintsとprobabilistic output constraintsを混同しません。
- joint q-batch supportとsequential optimization supportは別capabilityです。
- mixed variableをcontinuous relaxationとして黙って扱いません。
- one-shot acquisitionはauxiliary variablesを含むため通常のq-batchだけでは互換性を判断できません。
- TuRBOはoptimizer backend名ではなくstateful search geometryです。
- unsupported combinationはsilent approximationではなくfail-fastさせます。

## 13. Related documentation

- `docs-internal/architecture/optimization/optimizer-architecture.md`
- `docs-internal/contracts/optimization/acquisition-integration.md`
- `docs/theory/optimization/12_selection_guide.md`
- `docs/theory/optimization/05_constraints.md`
- `docs/theory/optimization/08_batch_one_shot.md`

Phase 18以降ではBatch / Async / Fantasization、Initialization、TuRBOなど、ここで分離したspecialized semanticsを個別に扱います。

## 14. 既存APIの使い分け

| 目的 | 既存API | 重要な設定 |
|---|---|---|
| GPの学習と再学習 | `SingleTaskGP.fit()` | 観測データを更新 |
| 改善に基づく獲得関数 | `qLogExpectedImprovement` | 観測最良値 `best_f` |
| 連続候補の最適化 | `optimize_acqf(optimizer="botorch")` | `num_restarts`, `raw_samples` |
| 2点の同時提案 | `optimize_acqf(q=2)` | joint q-batch |
| 入力固定 | `fixed_features` | 固定する変数番号と値 |
| 入力・候補間制約 | `CandidateConstraints` | 制約係数と添字 |
| 勾配を使わない探索 | `optimize_acqf(optimizer="sobol")` | `raw_samples`, `seed` |
| 最適化器の対応状況 | `get_optimizer_capabilities` | q-batch、制約、固定変数 |

**ポイント：** まず獲得関数が表す目的を定め、次に探索空間の型・制約・q-batchの要件に適した最適化器を選びます。